In [ ]:
import pandas as pd
c = pd.read_csv("customers.csv", parse_dates=["signup_date"])
o = pd.read_csv("orders.csv", parse_dates=["order_date", "updated_at"])
a = pd.read_csv("customer_activity.csv", parse_dates=["event_date"])
for name, d in {"customers": c, "orders": o, "activity": a}.items():
    print(name, d.shape, "duplicate rows:", d.duplicated().sum(), "nulls:", d.isna().sum().sum())
print("unique order_id:", o.order_id.nunique(), "| orders without customer:", (~o.customer_id.isin(c.customer_id)).sum())
c.head()

In [ ]:
c["phone_n"] = c.phone.str.replace(r"\D", "", regex=True)
c = c.sort_values("signup_date")
c["master_id"] = c.groupby("phone_n").customer_id.transform("first")
conf = c[c.groupby("master_id").city.transform("nunique") > 1]
print(c.master_id.nunique(), conf.master_id.nunique())

In [ ]:
t = o.amount.sum()
o = o.sort_values("updated_at").drop_duplicates("order_id", keep="last")
print(len(o), round(t - o.amount.sum(), 2))

In [ ]:
m = c.set_index("customer_id").master_id
o = o.assign(master_id=o.customer_id.map(m))
a = a.drop_duplicates()
a = a.assign(master_id=a.customer_id.map(m))

In [ ]:
d = c.drop_duplicates("master_id").set_index("master_id").join(o[o.status.eq("completed")].groupby("master_id").agg(orders=("order_id", "size"), spent=("amount", "sum"), last=("order_date", "max")))
d["events"] = a.groupby("master_id").size()
d.describe().round(1)

In [ ]:
print(d.spent.sum().round(2), o.dropna().query("status == 'completed'").amount.sum().round(2))

In [ ]:
x = pd.concat([conf.assign(issue="city_conflict"), o[o.master_id.isna()].assign(issue="orphan_order"), a[a.master_id.isna()].assign(issue="orphan_event")])
x.to_csv("exceptions_report.csv")
d.to_csv("customers_consolidated.csv")
c.to_csv("customers.csv")
x.issue.value_counts()

## Summary
- 500 customers from 542 records; $11,580 of duplicates removed.
- Input files generated with AI.